# Autism Facial Expression Recognition — Kaggle Pipeline

Stratified 5-Fold Cross-Validation comparing 12 baseline architectures against the proposed **CareFER** dual-stream model (VGG16-BN + DeiT-Small) on RAW facial expression images.

## How to run

1. Upload `dataset_clean/` to Kaggle as a dataset. The scripts auto-detect it under `/kaggle/input`.
2. Run **Cell 1** — trains 12 baseline models (5 folds each). Writes checkpoints and resume markers to `/kaggle/working/results/`.
3. Run **Cell 2** — trains the proposed CareFER model (same 5 folds).

Both cells resume automatically if re-run after a session stops — completed folds are skipped.

## Cross-session resume (when training spans multiple Kaggle sessions)

1. Before stopping: click **Stop** → then **Save Version → Quick Save**.
2. Before the next session: right panel → **Input → Add Data → Your Work** → find this notebook → add its output.
3. Run the notebook again — prior checkpoints are restored automatically at startup.

## Dataset

- 1,808 deduplicated RAW images across 6 emotion classes: `anger`, `fear`, `joy`, `natural`, `sadness`, `surprise`.
- No MTCNN or CLAHE preprocessing (it hurt accuracy).
- Class imbalance handled via `WeightedRandomSampler` (baselines) and `FocalLoss` alpha weights (proposed model).


In [1]:
import os
import shutil

os.makedirs("/kaggle/working/results", exist_ok=True)

for root, dirs, files in os.walk("/kaggle/input"):
    if "cv_done.json" in files:
        print(f"Found saved progress at: {root}")

        # Copy everything EXCEPT the massive .pth files!
        for src_dir, _, src_files in os.walk(root):
            rel_dir = os.path.relpath(src_dir, root)
            dst_dir = os.path.normpath(
                os.path.join("/kaggle/working/results", rel_dir)
            )
            os.makedirs(dst_dir, exist_ok=True)

            for f in src_files:
                if not f.endswith(".pth"):
                    shutil.copy2(
                        os.path.join(src_dir, f),
                        os.path.join(dst_dir, f)
                    )

        print("Successfully copied progress (ignored massive .pth files)!")
        break

Found saved progress at: /kaggle/input/datasets/mrsohel/autism-fer-saved-results/results
Successfully copied progress (ignored massive .pth files)!


In [2]:
"""
=============================================================
  Autism Facial Expression Recognition — Kaggle Pipeline
  Train 12 curated models with Stratified K-Fold Cross-Validation
  on the RAW dataset (no face-crop / CLAHE preprocessing).
=============================================================

Methodology fixes vs. the previous version:
  1. RAW images only — the MTCNN+CLAHE step measurably hurt accuracy
     (vgg16 F1 0.548->0.528, resnet50/vit_base collapsed). Removed.
  2. Single weighting — class imbalance is handled by WeightedRandomSampler
     ONLY. Class weights are no longer passed into FocalLoss/CE
     (the old code double-weighted, over-regularising small models).
  3. Lighter train-time augmentation — MixUp and RandomErasing removed.
     On ~2k images they added more noise than signal.
  4. Stratified K-fold CV — every image is predicted exactly once
     (out-of-fold), so per-class metrics on rare emotions (fear n~14)
     are no longer statistically meaningless. Results are mean +/- std.

Resumable: per-fold checkpoints + a cv_done.json marker are saved to
OUTPUT_DIR after every fold. If a Kaggle session times out, just re-run
the cell — already-completed (model, fold) pairs are skipped.
"""

import os, sys, json, time, math
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler
from torch.amp import autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
import pandas as pd

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: No GPU detected — training will be very slow")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DL_GENERATOR = torch.Generator().manual_seed(SEED)


def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    torch.manual_seed(SEED + worker_id)

# ---- Paths (Kaggle default) ----
def find_dataset_dir(hardcoded):
    """Locate the dataset root containing train/valid/test class folders."""
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return hardcoded
        
    for root, dirs, _ in os.walk(base):
        if all(split in dirs for split in ("train", "valid", "test")):
            print(f"[*] Auto-detected dataset at {root}")
            return root
            
    print(f"[!] Dataset not found under {base}; using hardcoded path")
    return hardcoded


DATA_DIR   = find_dataset_dir("/kaggle/input/datasets/mrsohel/dataset-clean")
OUTPUT_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[*] DATA_DIR = {DATA_DIR}")


def restore_from_prior_session(output_dir):
    """Copy checkpoint files from a prior session's Kaggle output into output_dir.

    Workflow:
      Session 1 finishes (or times out) → Kaggle saves /kaggle/working/ as
      notebook output.  Before Session 2, the user adds that output as an
      input dataset.  This function detects any such prior-output dataset
      (it has cv_done.json at root or inside a 'results/' sub-folder) and
      copies everything into output_dir so the in-session resume logic finds
      it at the expected paths.

    NOTE: .pth checkpoint files are intentionally skipped — they are large
    (100-400 MB each) and NOT needed for the resume logic (only .npy OOF
    arrays and .json metrics are checked). Skipping them keeps this fast.
    """
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return
    for entry in os.listdir(base):
        entry_path = os.path.join(base, entry)
        if not os.path.isdir(entry_path):
            continue
        # Accept both flat layout (cv_done.json at root) and nested (results/cv_done.json)
        candidate_roots = [entry_path, os.path.join(entry_path, "results")]
        for croot in candidate_roots:
            if os.path.exists(os.path.join(croot, "cv_done.json")):
                print(f"[*] Found prior session output at {croot} — restoring to {output_dir}")
                import shutil
                copied, skipped_pth = 0, 0
                for item in os.listdir(croot):
                    src = os.path.join(croot, item)
                    dst = os.path.join(output_dir, item)
                    if os.path.isdir(src):
                        if not os.path.exists(dst):
                            os.makedirs(dst, exist_ok=True)
                        # Merge: only copy files that don't already exist; skip .pth
                        for root, dirs, files in os.walk(src):
                            rel = os.path.relpath(root, src)
                            dest_root = os.path.join(dst, rel)
                            os.makedirs(dest_root, exist_ok=True)
                            for f in files:
                                if f.endswith(".pth"):
                                    skipped_pth += 1
                                    continue  # skip — not needed for resume
                                dest_f = os.path.join(dest_root, f)
                                if not os.path.exists(dest_f):
                                    shutil.copy2(os.path.join(root, f), dest_f)
                                    copied += 1
                                    if copied % 10 == 0:
                                        print(f"    ... copied {copied} files so far (skipped {skipped_pth} .pth)")
                    else:
                        if not os.path.exists(dst) and not item.endswith(".pth"):
                            shutil.copy2(src, dst)
                            copied += 1
                print(f"[*] Restore complete: {copied} files copied, {skipped_pth} large .pth files skipped.")
                return  # only restore from the first matching prior output


restore_from_prior_session(OUTPUT_DIR)

# ---- Hyperparameters ----
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 80
N_FOLDS    = 5          # stratified K-fold CV (set to 3 to save GPU time)
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 15
EMA_DECAY     = 0.999
NUM_WORKERS   = 2 if sys.platform == "linux" else 0  # Windows spawn breaks multiprocessing loaders
# Bump when the OOF evaluation / training methodology changes. Persistent cv_metrics.json
# written by an older version is incompatible and must be deleted before re-running.
PIPELINE_VERSION = "v3-emabn"

CLASS_NAMES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# ---- Models to train ----
# Curated SOTA set for the supervisor comparison — one representative per
# architectural family: classic CNNs, a modern CNN (ConvNeXt), ViTs/hybrids,
# and existing CNN-ViT hybrids (MaxViT, EfficientFormer) that the Proposed
# hybrid must beat. All run at 224 input for a fair comparison.
EXPERIMENTS = [
    # Classic CNNs
    {"model": "vgg16",                        "loss": "focal",     "lr": 1e-3},
    {"model": "resnet50",                     "loss": "focal",     "lr": 1e-3},
    {"model": "densenet121",                  "loss": "focal",     "lr": 1e-3},
    {"model": "efficientnet_b0",              "loss": "focal",     "lr": 1e-3},
    {"model": "mobilenetv2_100",              "loss": "focal",     "lr": 1e-3},
    # Modern CNN (current SOTA dense family)
    {"model": "convnext_tiny",                "loss": "focal",     "lr": 1e-3},
    # Vision Transformers (lower LR to prevent collapse)
    {"model": "deit_small_patch16_224",       "loss": "ce_smooth", "lr": 1e-4},
    {"model": "vit_base_patch16_224",         "loss": "ce_smooth", "lr": 1e-4},
    {"model": "swin_tiny_patch4_window7_224", "loss": "ce_smooth", "lr": 1e-4},
    {"model": "swin_base_patch4_window7_224", "loss": "ce_smooth", "lr": 1e-4},
    # Existing CNN-ViT hybrids (motivates the Proposed hybrid architecture)
    {"model": "maxvit_tiny_rw_224",           "loss": "ce_smooth", "lr": 1e-4},
    {"model": "efficientformer_l1",           "loss": "ce_smooth", "lr": 1e-4},
]

# ==============================================================================
# Dataset — all splits merged, CV partitions at runtime
# ==============================================================================
class FacialExpressionDataset(Dataset):
    def __init__(self, root_dir, samples, labels, transform=None):
        self.root_dir = root_dir
        self.samples = samples
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image = Image.open(self.samples[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


def build_full_dataset(root_dir):
    """Collect every image across train/valid/test into one list."""
    samples, labels = [], []
    for split in ("train", "valid", "test"):
        for class_name in CLASS_NAMES:
            class_dir = Path(root_dir) / split / class_name
            if not class_dir.exists():
                continue
            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".tiff"):
                    samples.append(str(img_path))
                    labels.append(CLASS_TO_IDX[class_name])
    return samples, labels


def get_train_transforms(img_size=IMG_SIZE):
    # Aspect-preserving: RandomResizedCrop replaces the old Resize((s,s)) squash.
    # MixUp / RandomErasing removed — too destructive on a small dataset.
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0), ratio=(0.75, 1.333)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def get_val_transforms(img_size=IMG_SIZE):
    # Preserve aspect ratio (resize long side, then center-crop) instead of the
    # old Resize((s, s)) squash — matches ImageNet-style fine-tuning.
    rs = int(img_size * 1.143)
    return transforms.Compose([
        transforms.Resize(rs),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def get_tta_transforms(img_size=IMG_SIZE):
    # 5-view TTA for the final OOF pass, so baselines get the same inference
    # treatment as Proposed-Model (5-view TTA) and the comparison stays fair.
    # All views are deterministic (no random sampling) so the reported OOF
    # numbers are byte-stable across runs.
    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    rs = int(img_size * 1.143)
    base = [transforms.Resize(rs), transforms.CenterCrop(img_size)]
    zoom = int(img_size * 1.10)
    return [
        transforms.Compose(base + [transforms.ToTensor(), norm]),
        transforms.Compose(base + [transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((img_size, img_size)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize(zoom), transforms.CenterCrop(img_size), transforms.ToTensor(), norm]),
    ]


def make_loaders(samples, labels, train_idx, val_idx, batch_size=BATCH_SIZE, img_size=IMG_SIZE):
    train_ds = FacialExpressionDataset(DATA_DIR,
                                       [samples[i] for i in train_idx],
                                       [labels[i] for i in train_idx],
                                       get_train_transforms(img_size))
    val_ds   = FacialExpressionDataset(DATA_DIR,
                                       [samples[i] for i in val_idx],
                                       [labels[i] for i in val_idx],
                                       get_val_transforms(img_size))

    # SINGLE weighting mechanism: sampler only (no class weights in the loss).
    # Cap the weight ratio so rare classes are still oversampled without being
    # shown the same ~68 images many times per epoch (which caused overfitting).
    counts = Counter(train_ds.labels)
    inv_freq = [1.0 / counts[label] for label in train_ds.labels]
    w_min = min(inv_freq)
    sample_weights = [min(w, w_min * 9.0) for w in inv_freq]  # ~max 9x boost
    sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              worker_init_fn=seed_worker, generator=DL_GENERATOR)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              worker_init_fn=seed_worker, generator=DL_GENERATOR)
    return train_loader, val_loader, val_ds


# ==============================================================================
# Losses
# ==============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()


def get_loss_fn(loss_type):
    if loss_type == "focal":
        return FocalLoss(gamma=2.0)
    elif loss_type == "ce_smooth":
        return nn.CrossEntropyLoss(label_smoothing=0.1)
    return nn.CrossEntropyLoss()


# ==============================================================================
# Model factory
# ==============================================================================
MODEL_CONFIGS = {
    "vgg16": {"timm": "vgg16_bn", "size": 224},
    "densenet121": {"timm": "densenet121", "size": 224},
    "efficientnet_b0": {"timm": "efficientnet_b0", "size": 224},
    "mobilenetv2_100": {"timm": "mobilenetv2_100", "size": 224},
    "resnet50": {"timm": "resnet50", "size": 224},
    "convnext_tiny": {"timm": "convnext_tiny", "size": 224},
    "deit_small_patch16_224": {"timm": "deit_small_patch16_224", "size": 224},
    "vit_base_patch16_224": {"timm": "vit_base_patch16_224.augreg_in21k", "size": 224},
    "swin_base_patch4_window7_224": {"timm": "swin_base_patch4_window7_224", "size": 224},
    "swin_tiny_patch4_window7_224": {"timm": "swin_tiny_patch4_window7_224", "size": 224},
    "maxvit_tiny_rw_224": {"timm": "maxvit_tiny_rw_224", "size": 224},
    "efficientformer_l1": {"timm": "efficientformer_l1", "size": 224},
}


def get_model(name, pretrained=True):
    cfg = MODEL_CONFIGS[name]
    try:
        try:
            model = timm.create_model(cfg["timm"], pretrained=pretrained, num_classes=NUM_CLASSES,
                                      drop_rate=0.3, drop_path_rate=0.2)
        except TypeError:
            model = timm.create_model(cfg["timm"], pretrained=pretrained, num_classes=NUM_CLASSES)
    except RuntimeError as e:
        if pretrained and "pretrained" in str(e).lower():
            print(f"Warning: No pretrained weights for {name}. Using random init.")
            return get_model(name, pretrained=False)
        raise
    return model, cfg["size"]


# ==============================================================================
# Training helpers
# ==============================================================================
class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = (1 - self.decay) * p.data + self.decay * self.shadow[n]

    def apply_shadow(self):
        self.backup = {n: p.data.clone() for n, p in self.model.named_parameters() if p.requires_grad}
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                p.data = self.shadow[n]

    def restore(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                p.data = self.backup[n]


def train_one_epoch(model, loader, criterion, optimizer, scaler, ema):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=DEVICE.type, dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        if ema:
            ema.update()

        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * images.size(0)

    return total_loss / total, correct / total if total else 0


@torch.no_grad()
def evaluate(model, loader, ema=None):
    if ema:
        ema.apply_shadow()
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    if ema:
        ema.restore()
    return [int(p) for p in all_preds], [int(l) for l in all_labels], np.array(all_probs)


@torch.no_grad()
def evaluate_tta(dataset, model, device, tta_views, batch_size=BATCH_SIZE):
    """Final OOF pass with multi-view TTA, mirroring Proposed-Model's inference.

    Processes each view over the whole (batched) validation set so the forward
    passes scale with len(views) instead of len(dataset) * len(views).
    """
    model.eval()
    n = len(dataset)
    all_probs = np.zeros((n, NUM_CLASSES), dtype=np.float64)
    for view in tta_views:
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch = torch.stack(
                [view(Image.open(dataset.samples[i]).convert("RGB")) for i in range(start, end)]
            ).to(device)
            logits = model(batch)
            all_probs[start:end] += F.softmax(logits, dim=1).cpu().numpy()
    all_probs /= len(tta_views)
    preds = all_probs.argmax(1).tolist()
    return preds, [int(l) for l in dataset.labels], all_probs


def compute_metrics(y_true, y_pred):
    labels_list = list(range(NUM_CLASSES))
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0, labels=labels_list),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0, labels=labels_list),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0, labels=labels_list),
        "per_class_f1": dict(zip(CLASS_NAMES, [float(f) for f in f1_score(y_true, y_pred, average=None, zero_division=0, labels=labels_list)])),
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0, labels=labels_list),
    }


def fold_mean(fold_metrics):
    """Mean +/- std of a metric key across completed folds."""
    mean = {}
    for key in ("accuracy", "f1_macro", "precision_macro", "recall_macro"):
        values = [f[key] for f in fold_metrics]
        mean[key] = float(np.mean(values))
        mean[key + "_std"] = float(np.std(values))
    return mean


# ==============================================================================
# Resume support
# ==============================================================================
DONE_FILE = os.path.join(OUTPUT_DIR, "cv_done.json")
done = {}
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f:
        done = json.load(f)
    print(f"[*] Found {len(done)} model(s) with completed folds — resuming.")


def fold_done(name, fold):
    return name in done and fold in done[name]


def mark_done(name, fold):
    done.setdefault(name, []).append(fold)
    with open(DONE_FILE, "w") as f:
        json.dump(done, f, indent=2)


# ==============================================================================
# Cross-validation splits (shared with the Proposed-Model script)
# ==============================================================================
print("[*] Building full dataset (train+valid+test merged) ...")
samples, labels = build_full_dataset(DATA_DIR)
print(f"[*] Total images: {len(samples)} | Per-class: {dict(Counter(labels))}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(samples, labels))

# Persist the fold assignment so run_proposed_model.py uses identical folds.
fold_id_by_path = {}
for fold_idx, (_, val_idx) in enumerate(folds):
    for i in val_idx:
        fold_id_by_path[samples[i]] = fold_idx
with open(os.path.join(OUTPUT_DIR, "fold_id_by_path.json"), "w") as f:
    json.dump(fold_id_by_path, f)
print(f"[*] Saved fold assignment to {OUTPUT_DIR}/fold_id_by_path.json")


# ==============================================================================
# Per-fold training
# ==============================================================================
def run_fold(name, exp, fold, train_idx, val_idx):
    print(f"\n{'='*60}")
    print(f"  {name} | Loss: {exp['loss']} | Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*60}")

    model, input_size = get_model(name, pretrained=True)
    model = model.to(DEVICE)

    train_loader, val_loader, val_dataset = make_loaders(samples, labels, train_idx, val_idx, img_size=input_size)

    criterion = get_loss_fn(exp["loss"])

    # Differential learning rates (backbone 0.1x, head 1x)
    backbone, head = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in n for k in ("classifier", "head", "fc")):
            head.append(p)
        else:
            backbone.append(p)

    exp_lr = exp.get("lr", LEARNING_RATE)
    optimizer = torch.optim.AdamW([
        {"params": backbone, "lr": exp_lr * 0.1},
        {"params": head, "lr": exp_lr},
    ], weight_decay=WEIGHT_DECAY)
    # Transformers collapse under CosineAnnealingWarmRestarts' periodic LR spikes —
    # give them warmup + cosine (same schedule family as Proposed-Model).
    if exp.get("loss") == "ce_smooth":
        _warmup = 5
        def _lr_lambda(ep):
            if ep < _warmup:
                return (ep + 1) / _warmup
            return 0.5 * (1 + math.cos(math.pi * (ep - _warmup) / max(1, NUM_EPOCHS - _warmup)))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_lambda)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = GradScaler(enabled=(DEVICE.type == "cuda"))
    ema = EMA(model, EMA_DECAY)

    best_f1 = 0.0
    patience_counter = 0
    ckpt_path = f"{OUTPUT_DIR}/{name}/fold{fold+1}_best.pth"
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    t0 = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, ema)
        scheduler.step()
        preds, val_labels, _ = evaluate(model, val_loader, ema)
        val_m = compute_metrics(val_labels, preds)

        elapsed = time.time() - t0
        print(f"  Epoch [{epoch:2d}/{NUM_EPOCHS}] "
              f"| Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:5.1f}% "
              f"| Val Acc: {val_m['accuracy']*100:5.1f}% | Val Macro-F1: {val_m['f1_macro']:.4f} "
              f"| Time: {elapsed/60:.1f}m")

        if val_m["f1_macro"] > best_f1:
            best_f1 = val_m["f1_macro"]
            patience_counter = 0
            torch.save({"epoch": epoch, "state_dict": model.state_dict(),
                        "ema": ema.shadow, "args": exp}, ckpt_path)
            print(f"    >> New best fold F1: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"    >> Early stopping at epoch {epoch}")
                break

    # --- Out-of-fold evaluation with the EMA weights + 5-view TTA ---
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    ema.shadow = ckpt["ema"]
    ema.apply_shadow()
    preds, val_labels, probs = evaluate_tta(val_dataset, model, DEVICE, get_tta_transforms(input_size))
    ema.restore()
    fold_m = compute_metrics(val_labels, preds)
    print(f"  FOLD {fold+1} OOF (TTA) — Acc: {fold_m['accuracy']:.4f} | F1: {fold_m['f1_macro']:.4f}")

    # Cleanup (Kaggle DataLoader worker leaks)
    del model, optimizer, scheduler, scaler, criterion, ema
    del train_loader, val_loader
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    return fold_m, preds, val_labels, probs


# ==============================================================================
# Run all models over all folds (OOF arrays + metrics persisted after every fold
# so a resumed session always works with the full set of completed folds)
# ==============================================================================
def load_npy(path, empty_shape):
    if os.path.exists(path):
        arr = np.load(path)
        return arr if arr.ndim > 0 else np.empty(empty_shape)
    return np.empty(empty_shape)


for exp in EXPERIMENTS:
    name = exp["model"]
    print(f"\n{'#'*70}\n# {name}\n{'#'*70}")

    model_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(model_dir, exist_ok=True)

    oof_preds = load_npy(f"{model_dir}/oof_preds.npy", (0,))
    oof_labels = load_npy(f"{model_dir}/oof_labels.npy", (0,))
    oof_probs = load_npy(f"{model_dir}/oof_probs.npy", (0, NUM_CLASSES))
    fold_metrics = []
    metrics_path = f"{model_dir}/cv_metrics.json"
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            _existing = json.load(f)
        fold_metrics = _existing["folds"]
        if _existing.get("pipeline_version") != PIPELINE_VERSION:
            print(f"[!] {name}: existing cv_metrics.json was produced by another eval "
                  f"pipeline (found '{_existing.get('pipeline_version', 'v1')}' != "
                  f"'{PIPELINE_VERSION}'). Resmuming would MIX incompatible metrics — "
                  f"delete {model_dir} before re-running or results will be invalid.")

    ran_any = False
    for fold, (train_idx, val_idx) in enumerate(folds):
        if fold_done(name, fold):
            print(f"  Fold {fold+1} already done — skipping.")
            continue
        ran_any = True

        fold_m, preds, val_labels, probs = run_fold(name, exp, fold, train_idx, val_idx)
        fold_metrics.append({"fold": fold, **fold_m})
        oof_preds = np.concatenate([oof_preds, np.array(preds, dtype=int)])
        oof_labels = np.concatenate([oof_labels, np.array(val_labels, dtype=int)])
        oof_probs = np.concatenate([oof_probs, probs], axis=0)

        # incremental persistence (safe across session timeouts)
        np.save(f"{model_dir}/oof_preds.npy", oof_preds)
        np.save(f"{model_dir}/oof_labels.npy", oof_labels)
        np.save(f"{model_dir}/oof_probs.npy", oof_probs)
        with open(metrics_path, "w") as f:
            json.dump({"folds": fold_metrics}, f, indent=2)
        mark_done(name, fold)

    if not ran_any:
        # resumed session where all folds were already completed
        if not fold_metrics:
            raise RuntimeError(f"{name}: cv_metrics.json missing but folds marked done — "
                               "results directory is inconsistent.")
        with open(metrics_path) as f:
            oof_m = json.load(f)
        if "mean" not in oof_m:
            # Session may have died before the final full write; recompute it.
            oof_m["mean"] = fold_mean(fold_metrics)
            oof_m["n_folds"] = len(fold_metrics)
    else:
        oof_m = compute_metrics(oof_labels.tolist(), oof_preds.tolist())
        oof_m["mean"] = fold_mean(fold_metrics)
        oof_m["n_folds"] = len(fold_metrics)
        oof_m["folds"] = fold_metrics
        oof_m["pipeline_version"] = PIPELINE_VERSION
        oof_m["params"] = sum(p.numel() for p in get_model(name, pretrained=False)[0].parameters())
        with open(metrics_path, "w") as f:
            json.dump(oof_m, f, indent=2)

    print(f"\n  {name} CV mean ({oof_m.get('n_folds', len(fold_metrics))} folds) — "
          f"Acc: {oof_m['mean']['accuracy']:.4f}+/-{oof_m['mean']['accuracy_std']:.4f} | "
          f"F1: {oof_m['mean']['f1_macro']:.4f}+/-{oof_m['mean']['f1_macro_std']:.4f}")

# ==============================================================================
# Cross-model comparison (from saved cv_metrics.json so figures work on resume)
# ==============================================================================
def load_oof(name):
    model_dir = os.path.join(OUTPUT_DIR, name)
    return (np.load(f"{model_dir}/oof_preds.npy"),
            np.load(f"{model_dir}/oof_probs.npy"),
            np.load(f"{model_dir}/oof_labels.npy"))

all_results = {}
for exp in EXPERIMENTS:
    name = exp["model"]
    with open(f"{OUTPUT_DIR}/{name}/cv_metrics.json") as f:
        m = json.load(f)
    m["args"] = exp
    all_results[name] = m

print(f"\n{'='*70}")
print("  FINAL MODEL COMPARISON (Stratified K-Fold CV)")
print(f"{'='*70}")
header = f"{'Model':<35} {'Acc':>10} {'F1':>10} {'Prec':>10} {'Rec':>10} {'Folds':>6}"
print(header)
print("-" * len(header))
for name in sorted(all_results.keys(), key=lambda k: all_results[k]["mean"]["f1_macro"], reverse=True):
    r = all_results[name]["mean"]
    n_folds = all_results[name].get("n_folds", len(all_results[name].get("folds", [])))
    print(f"{name:<35} {r['accuracy']:>6.4f}+/-{r['accuracy_std']:.3f} "
          f"{r['f1_macro']:>6.4f}+/-{r['f1_macro_std']:.3f} "
          f"{r['precision_macro']:>6.4f}+/-{r['precision_macro_std']:.3f} "
          f"{r['recall_macro']:>6.4f}+/-{r['recall_macro_std']:.3f} {n_folds:>6}")

# ==============================================================================
# Paper figures (CV-aware)
# ==============================================================================
COMPARISON_DIR = os.path.join(OUTPUT_DIR, "paper_figures")
os.makedirs(COMPARISON_DIR, exist_ok=True)
models_sorted = sorted(all_results.keys(), key=lambda k: all_results[k]["mean"]["f1_macro"], reverse=True)
top_5 = models_sorted[:min(5, len(models_sorted))]

# 1. Grouped bar chart with error bars
rows = []
for m in models_sorted:
    r = all_results[m]["mean"]
    for metric, key in [("Accuracy", "accuracy"), ("F1-Macro", "f1_macro"),
                        ("Precision", "precision_macro"), ("Recall", "recall_macro")]:
        rows.append({"Model": m, "Metric": metric, "Score": r[key],
                     "Std": r[key + "_std"]})
df_metrics = pd.DataFrame(rows)
fig1, ax1 = plt.subplots(figsize=(12, 6))
sns.barplot(data=df_metrics, x="Model", y="Score", hue="Metric", palette="Set2", ax=ax1)
# overlay std as error caps
for i, metric in enumerate(["Accuracy", "F1-Macro", "Precision", "Recall"]):
    for j, m in enumerate(models_sorted):
        row = df_metrics[(df_metrics["Metric"] == metric) & (df_metrics["Model"] == m)].iloc[0]
        ax1.errorbar(x=j + (i - 1.5) * 0.2, y=row["Score"], yerr=row["Std"],
                     fmt="none", c="black", capsize=2, linewidth=0.8)
ax1.set_ylim(0, 1.0)
ax1.set_title("Model Comparison (Stratified K-Fold CV) - Mean +/- Std")
plt.tight_layout()
plt.savefig(f"{COMPARISON_DIR}/1_cv_grouped_bar_metrics.png", dpi=300)
plt.close(fig1)

# 2. Box plot of fold-level F1 across models
fold_rows = []
for exp in EXPERIMENTS:
    name = exp["model"]
    with open(f"{OUTPUT_DIR}/{name}/cv_metrics.json") as f:
        m = json.load(f)
    # fold-level F1 is not in cv_metrics.json; reconstruct from per-fold jsons is skipped.
    # Use mean/std as pseudo-box for robustness across resume.
    fold_rows.append({"Model": name, "F1": m["mean"]["f1_macro"],
                      "lower": m["mean"]["f1_macro"] - m["mean"]["f1_macro_std"],
                      "upper": m["mean"]["f1_macro"] + m["mean"]["f1_macro_std"]})
df_f1 = pd.DataFrame(fold_rows)
fig2, ax2 = plt.subplots(figsize=(12, 6))
sns.barplot(data=df_f1, x="Model", y="F1", palette="coolwarm", ax=ax2)
for i, row in df_f1.iterrows():
    ax2.errorbar(x=i, y=row["F1"], yerr=[[row["F1"] - row["lower"]], [row["upper"] - row["F1"]]],
                 fmt="none", c="black", capsize=3)
ax2.set_ylabel("Macro F1 (mean +/- std across folds)")
ax2.set_ylim(0, 0.8)
plt.tight_layout()
plt.savefig(f"{COMPARISON_DIR}/2_cv_f1_comparison.png", dpi=300)
plt.close(fig2)

# 3. OOF macro ROC (top 5)
fig3, ax3 = plt.subplots(figsize=(10, 8))
for m in top_5:
    preds, probs, labels = load_oof(m)
    Y_bin = label_binarize(labels, classes=list(range(NUM_CLASSES)))
    fpr, tpr, _ = roc_curve(Y_bin.ravel(), probs.ravel())
    macro_auc = auc(fpr, tpr)
    ax3.plot(fpr, tpr, lw=2, label=f"{m} (AUC = {macro_auc:.3f})")
ax3.plot([0, 1], [0, 1], 'k--', lw=2)
ax3.set_xlabel("False Positive Rate"); ax3.set_ylabel("True Positive Rate")
ax3.set_title("Macro-Average OOF ROC Curve (Top 5 Models)")
ax3.legend(loc="lower right"); ax3.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{COMPARISON_DIR}/3_roc_curve.png", dpi=300); plt.close(fig3)

# 4. OOF Precision-Recall (top 5)
fig4, ax4 = plt.subplots(figsize=(10, 8))
for m in top_5:
    preds, probs, labels = load_oof(m)
    Y_bin = label_binarize(labels, classes=list(range(NUM_CLASSES)))
    prec, rec, _ = precision_recall_curve(Y_bin.ravel(), probs.ravel())
    ap = average_precision_score(Y_bin, probs, average="macro")
    ax4.plot(rec, prec, lw=2, label=f"{m} (AP = {ap:.3f})")
ax4.set_xlabel("Recall"); ax4.set_ylabel("Precision")
ax4.set_title("Macro-Average OOF Precision-Recall Curve (Top 5 Models)")
ax4.legend(loc="lower left"); ax4.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{COMPARISON_DIR}/4_pr_curve.png", dpi=300); plt.close(fig4)

# 5. Radar chart (means)
categories = ["Accuracy", "F1-Macro", "Precision", "Recall"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
fig5, ax5 = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax5.set_theta_offset(np.pi / 2); ax5.set_theta_direction(-1)
plt.xticks(angles[:-1], categories)
plt.yticks([0.2, 0.4, 0.6, 0.8], ["0.2", "0.4", "0.6", "0.8"], color="grey", size=8)
plt.ylim(0, 1)
for m in top_5:
    r = all_results[m]["mean"]
    values = [r["accuracy"], r["f1_macro"], r["precision_macro"], r["recall_macro"]]
    values += values[:1]
    ax5.plot(angles, values, linewidth=2, label=m)
    ax5.fill(angles, values, alpha=0.1)
plt.title("Radar Chart - Top 5 Models (CV means)", y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.savefig(f"{COMPARISON_DIR}/5_radar_chart.png", dpi=300, bbox_inches="tight"); plt.close(fig5)

# 6. OOF confusion matrix heatmap for the best model
best_model = models_sorted[0]
preds, probs, labels = load_oof(best_model)
cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
fig6, ax6 = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax6, vmin=0, vmax=1)
ax6.set_xlabel("Predicted"); ax6.set_ylabel("True")
ax6.set_title(f"{best_model} — Out-of-Fold Confusion Matrix")
plt.tight_layout(); plt.savefig(f"{COMPARISON_DIR}/6_best_model_oof_cm.png", dpi=300); plt.close(fig6)

# 7. Model prediction correlation heatmap (OOF, ensemble diversity)
preds_dict = {}
for m in models_sorted:
    p, _, _ = load_oof(m)
    preds_dict[m] = p
df_preds = pd.DataFrame(preds_dict)
corr = df_preds.corr(method="spearman").fillna(0)
fig7, ax7 = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap="coolwarm", vmin=0, vmax=1, ax=ax7)
ax7.set_title("Model Prediction Correlation (Spearman, OOF)")
plt.tight_layout(); plt.savefig(f"{COMPARISON_DIR}/7_model_correlation_heatmap.png", dpi=300); plt.close(fig7)

print(f"\nAll CV results and paper-ready figures saved to {COMPARISON_DIR}/")
print("Download the output dataset from the Kaggle 'Output' tab.")


PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
[*] Auto-detected dataset at /kaggle/input/datasets/mrsohel/dataset-clean/dataset_clean
[*] DATA_DIR = /kaggle/input/datasets/mrsohel/dataset-clean/dataset_clean
[*] Found 9 model(s) with completed folds — resuming.
[*] Building full dataset (train+valid+test merged) ...
[*] Total images: 1808 | Per-class: {0: 167, 1: 68, 2: 843, 3: 201, 4: 404, 5: 125}
[*] Saved fold assignment to /kaggle/working/results/fold_id_by_path.json

######################################################################
# vgg16
######################################################################
  Fold 1 already done — skipping.
  Fold 2 already done — skipping.
  Fold 3 already done — skipping.
  Fold 4 already done — skipping.
  Fold 5 already done — skipping.

  vgg16 CV mean (5 folds) — Acc: 0.7135+/-0.0140 | F1: 0.6114+/-0.0173

######################################################################
# resnet50
###############################################

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/tmp/ipykernel_23/2946703354.py:551: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE.type == "cuda"))


  Epoch [ 1/80] | Train Loss: 1.8123 | Train Acc:  18.6% | Val Acc:   9.7% | Val Macro-F1: 0.0685 | Time: 0.2m
    >> New best fold F1: 0.0685
  Epoch [ 2/80] | Train Loss: 1.6989 | Train Acc:  29.4% | Val Acc:  11.9% | Val Macro-F1: 0.0804 | Time: 0.5m
    >> New best fold F1: 0.0804
  Epoch [ 3/80] | Train Loss: 1.5453 | Train Acc:  39.9% | Val Acc:  16.6% | Val Macro-F1: 0.1047 | Time: 0.7m
    >> New best fold F1: 0.1047
  Epoch [ 4/80] | Train Loss: 1.4244 | Train Acc:  48.8% | Val Acc:  23.0% | Val Macro-F1: 0.1439 | Time: 0.9m
    >> New best fold F1: 0.1439
  Epoch [ 5/80] | Train Loss: 1.2637 | Train Acc:  59.4% | Val Acc:  29.4% | Val Macro-F1: 0.2218 | Time: 1.1m
    >> New best fold F1: 0.2218
  Epoch [ 6/80] | Train Loss: 1.1822 | Train Acc:  63.4% | Val Acc:  35.7% | Val Macro-F1: 0.2965 | Time: 1.3m
    >> New best fold F1: 0.2965
  Epoch [ 7/80] | Train Loss: 1.0918 | Train Acc:  67.2% | Val Acc:  44.6% | Val Macro-F1: 0.3580 | Time: 1.6m
    >> New best fold F1: 0.3580

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

...

  Epoch [ 1/80] | Train Loss: 1.8352 | Train Acc:  19.9% | Val Acc:  27.7% | Val Macro-F1: 0.1462 | Time: 0.2m
    >> New best fold F1: 0.1462
  Epoch [ 2/80] | Train Loss: 1.8271 | Train Acc:  18.6% | Val Acc:  45.4% | Val Macro-F1: 0.1122 | Time: 0.4m
  Epoch [ 3/80] | Train Loss: 1.8044 | Train Acc:  20.8% | Val Acc:  44.3% | Val Macro-F1: 0.1385 | Time: 0.6m
  Epoch [ 4/80] | Train Loss: 1.7935 | Train Acc:  21.1% | Val Acc:  43.8% | Val Macro-F1: 0.1131 | Time: 0.8m
  Epoch [ 5/80] | Train Loss: 1.7753 | Train Acc:  23.8% | Val Acc:   7.2% | Val Macro-F1: 0.0259 | Time: 1.0m
  Epoch [ 6/80] | Train Loss: 1.7332 | Train Acc:  27.6% | Val Acc:  46.5% | Val Macro-F1: 0.1059 | Time: 1.2m
  Epoch [ 7/80] | Train Loss: 1.6949 | Train Acc:  29.7% | Val Acc:  44.3% | Val Macro-F1: 0.1340 | Time: 1.4m
  Epoch [ 8/80] | Train Loss: 1.6661 | Train Acc:  35.1% | Val Acc:  22.2% | Val Macro-F1: 0.0605 | Time: 1.6m
  Epoch [ 9/80] | Train Loss: 1.6182 | Train Acc:  35.5% | Val Acc:   8.0% | Val

/tmp/ipykernel_23/2946703354.py:551: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE.type == "cuda"))


  Epoch [ 1/80] | Train Loss: 1.8423 | Train Acc:  16.5% | Val Acc:  11.4% | Val Macro-F1: 0.0783 | Time: 0.2m
    >> New best fold F1: 0.0783
  Epoch [ 2/80] | Train Loss: 1.8182 | Train Acc:  20.2% | Val Acc:  11.1% | Val Macro-F1: 0.0336 | Time: 0.4m
  Epoch [ 3/80] | Train Loss: 1.8118 | Train Acc:  20.6% | Val Acc:  11.4% | Val Macro-F1: 0.0651 | Time: 0.6m
  Epoch [ 4/80] | Train Loss: 1.7915 | Train Acc:  21.4% | Val Acc:   9.1% | Val Macro-F1: 0.0279 | Time: 0.8m
  Epoch [ 5/80] | Train Loss: 1.7543 | Train Acc:  24.3% | Val Acc:  11.1% | Val Macro-F1: 0.0333 | Time: 1.0m
  Epoch [ 6/80] | Train Loss: 1.7498 | Train Acc:  24.6% | Val Acc:  17.5% | Val Macro-F1: 0.0673 | Time: 1.2m
  Epoch [ 7/80] | Train Loss: 1.7113 | Train Acc:  28.7% | Val Acc:  22.4% | Val Macro-F1: 0.0611 | Time: 1.4m
  Epoch [ 8/80] | Train Loss: 1.6600 | Train Acc:  32.2% | Val Acc:  22.2% | Val Macro-F1: 0.0845 | Time: 1.6m
    >> New best fold F1: 0.0845
  Epoch [ 9/80] | Train Loss: 1.6562 | Train Acc

/tmp/ipykernel_23/2946703354.py:758: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_f1, x="Model", y="F1", palette="coolwarm", ax=ax2)



All CV results and paper-ready figures saved to /kaggle/working/results/paper_figures/
Download the output dataset from the Kaggle 'Output' tab.


In [3]:
"""
========================================================================================
Proposed-Model: Clinically-Aware Recalibrated Ensemble for Autism Facial Expression Recognition
========================================================================================
Self-contained Kaggle training + evaluation script for the proposed architecture,
adapted to Stratified K-Fold Cross-Validation on RAW images (no MTCNN/CLAHE):

1. Dual-Stream Backbone: VGG16 (Local Texture Expert) + DeiT-Small (Global Geometry Expert)
2. Feature Recalibration: Dual Squeeze-and-Excitation (SE) Channel Attention Blocks (r=16)
3. Training Stabilization: Exponential Moving Average (EMA, decay=0.999) + Focal Loss
4. Clinical Inference: 5-View Test-Time Augmentation (TTA, K=5)
5. Clinical Safety: Confidence Uncertainty Rejection Guardrail
6. Publication Figures: Confusion Matrix, Curves, Per-Class Metrics, Grad-CAM Heatmaps

Methodology fixes (match run_all_models.py):
- RAW images only (the MTCNN+CLAHE step hurt accuracy).
- Identical train-time augmentation as the baselines (fair comparison):
  no MixUp, no RandomErasing, no RandAugment.
- Class imbalance handled ONLY via FocalLoss class weights (single weighting).
- Stratified K-Fold CV using the SAME fold assignment as the baselines
  (fold_id_by_path.json written by run_all_models.py).
- Resumable: per-fold checkpoints + cv_done.json marker.

Run run_all_models.py FIRST (it writes fold_id_by_path.json), then this cell.
========================================================================================
"""

import os
import sys
import copy
import time
import math
import random
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler
from torch.amp import autocast
from torch.nn.utils import clip_grad_norm_
import torchvision.transforms as transforms
import timm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. HYPERPARAMETERS & CONFIGURATION
# ==============================================================================
SEED = 42
NUM_EPOCHS = 160      # Stage 1
STAGE2_EPOCHS = 20    # Stage 2 (frozen backbone)
BATCH_SIZE = 16
LEARNING_RATE = 1e-4  # Optimal for Hybrid Transformer-CNN architectures
WEIGHT_DECAY = 1e-4
EMA_DECAY = 0.999
TTA_VIEWS = 5
UNCERTAINTY_THRESH = 0.30  # Clinical rejection guardrail
IMG_SIZE = 224
PATIENCE = 20
N_FOLDS = 5           # MUST match run_all_models.py
NUM_WORKERS = 2 if sys.platform == "linux" else 0  # Windows spawn breaks multiprocessing loaders

_EXPECTED_CLASSES = ("anger", "fear", "joy", "natural", "sadness", "surprise")

def find_dataset_dir(hardcoded=None):
    """Locate the dataset root containing train/valid/test class folders."""
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return hardcoded
        
    for root, dirs, _ in os.walk(base):
        if all(split in dirs for split in ("train", "valid", "test")):
            print(f"[*] Auto-detected dataset at {root}")
            return root
            
    if hardcoded:
        print(f"[!] Dataset not found under {base}; using hardcoded path")
    return hardcoded

KAGGLE_MTCNN_DIR   = "/kaggle/working/dataset_mtcnn"
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/mrsohel/autism-dataset/dataset_clean"
LOCAL_DATASET_DIR  = r"C:\Users\mrsoh\Documents\Autism-Facial-Expression-Recognition\dataset_clean"

# RAW dataset first — the MTCNN variant hurt accuracy.
# Priority: auto-detect (most reliable on Kaggle) > hardcoded slug > local > MTCNN fallback
_auto_detected = find_dataset_dir() if os.path.isdir("/kaggle/input") else None
if _auto_detected:
    DATASET_DIR = _auto_detected
elif os.path.exists(KAGGLE_DATASET_DIR):
    DATASET_DIR = KAGGLE_DATASET_DIR
elif os.path.exists(LOCAL_DATASET_DIR):
    DATASET_DIR = LOCAL_DATASET_DIR
else:
    DATASET_DIR = KAGGLE_MTCNN_DIR
    print(f"[!] WARNING: All dataset paths failed — falling back to {KAGGLE_MTCNN_DIR} (likely empty!)"
          " Run Cell 2 (run_all_models.py) first and ensure the dataset slug is attached.")

OUTPUT_DIR = "/kaggle/working/results/proposed_model_proposed" if os.path.exists("/kaggle") else "./results/proposed_model_proposed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Also ensure the shared results root exists (fold_id_by_path.json lives there).
_SHARED_RESULTS = "/kaggle/working/results" if os.path.exists("/kaggle") else "./results"
os.makedirs(_SHARED_RESULTS, exist_ok=True)


def restore_from_prior_session(output_dir, shared_results_dir):
    """Copy checkpoint files from a prior session's Kaggle output into output_dir.

    Workflow:
      Session 1 finishes (or times out) → Kaggle saves /kaggle/working/ as
      notebook output.  Before Session 2, the user adds that output as an
      input dataset.  This function detects any such prior-output dataset
      (it has proposed_model_proposed/cv_done.json) and copies everything
      into output_dir so the in-session resume logic finds it at the expected
      paths.  It also restores fold_id_by_path.json to the shared results dir.
    """
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return
    import shutil

    def _merge_copy(src_dir, dst_dir):
        """Copy src_dir → dst_dir, skipping files that already exist."""
        os.makedirs(dst_dir, exist_ok=True)
        for item in os.listdir(src_dir):
            src = os.path.join(src_dir, item)
            dst = os.path.join(dst_dir, item)
            if os.path.isdir(src):
                _merge_copy(src, dst)
            elif not os.path.exists(dst):
                shutil.copy2(src, dst)

    for entry in os.listdir(base):
        entry_path = os.path.join(base, entry)
        if not os.path.isdir(entry_path):
            continue
        # Accept flat layout or nested results/ layout
        for croot in [entry_path, os.path.join(entry_path, "results")]:
            proposed_dir = os.path.join(croot, "proposed_model_proposed")
            if os.path.exists(os.path.join(proposed_dir, "cv_done.json")):
                print(f"[*] Found prior session output at {proposed_dir} — restoring to {output_dir}")
                _merge_copy(proposed_dir, output_dir)
                # Restore shared files (fold_id_by_path.json) from the parent results dir
                for shared_file in ("fold_id_by_path.json",):
                    src_shared = os.path.join(croot, shared_file)
                    dst_shared = os.path.join(shared_results_dir, shared_file)
                    if os.path.exists(src_shared) and not os.path.exists(dst_shared):
                        shutil.copy2(src_shared, dst_shared)
                        print(f"[*] Restored {shared_file} to {shared_results_dir}")
                print(f"[*] Restore complete.")
                return


restore_from_prior_session(OUTPUT_DIR, _SHARED_RESULTS)

# Bump when the training/eval methodology changes; warns if an old results dir is resumed.
PIPELINE_VERSION = "v3-emabn"
_ver_path = os.path.join(OUTPUT_DIR, "pipeline_version.json")
if os.path.exists(_ver_path):
    with open(_ver_path) as f:
        _old_ver = json.load(f).get("pipeline_version")
    if _old_ver != PIPELINE_VERSION:
        print(f"[!] {OUTPUT_DIR} was produced by pipeline '{_old_ver}' != "
              f"'{PIPELINE_VERSION}'. Resuming would MIX incompatible metrics — "
              "delete the results dir before re-running.")
with open(_ver_path, "w") as f:
    json.dump({"pipeline_version": PIPELINE_VERSION}, f, indent=2)

CLASSES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {cls_name: i for i, cls_name in enumerate(CLASSES)}


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
DL_GENERATOR = torch.Generator().manual_seed(SEED)


def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    torch.manual_seed(SEED + worker_id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Running on Device: {device} | Output Directory: {OUTPUT_DIR}")
print(f"[*] Dataset Directory: {DATASET_DIR}")


# ==============================================================================
# 2. DATASET — all splits merged, CV partitioned at runtime
# ==============================================================================
class AutismFERDataset(Dataset):
    def __init__(self, samples, labels, transform=None):
        self.samples = samples
        self.targets = labels
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.samples[idx]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, self.targets[idx]


def build_full_dataset(root_dir):
    """Collect every image across train/valid/test into one list."""
    samples, labels = [], []
    for split in ("train", "valid", "test"):
        for cls_name in CLASSES:
            cls_dir = Path(root_dir) / split / cls_name
            if not cls_dir.exists():
                continue
            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"):
                    samples.append(str(img_path))
                    labels.append(CLASS_TO_IDX[cls_name])
    return samples, labels


# Identical augmentation as the baselines (fair head-to-head comparison).
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.75, 1.333)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.143)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def get_tta_transforms():
    # Deterministic 5-view TTA, identical to the baseline script.
    norm = [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    rs = int(IMG_SIZE * 1.143)
    base = [transforms.Resize(rs), transforms.CenterCrop(IMG_SIZE)]
    zoom = int(IMG_SIZE * 1.10)
    return [
        transforms.Compose(base + norm),
        transforms.Compose(base + [transforms.RandomHorizontalFlip(p=1.0)] + norm),
        transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE))] + norm),
        transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(p=1.0)] + norm),
        transforms.Compose([transforms.Resize(zoom), transforms.CenterCrop(IMG_SIZE)] + norm),
    ]


# ==============================================================================
# 3. PROPOSED ARCHITECTURE: Proposed-Model (Dual-Stream SE Recalibrated Ensemble)
# ==============================================================================
class SqueezeExcitationBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        w = self.fc1(x)
        w = self.relu(w)
        w = self.fc2(w)
        w = self.sigmoid(w)
        return x * w


class CareFERModel(nn.Module):
    """Dual-stream: VGG16-BN spatial (forward_features + GAP, 512-d) + DeiT-S CLS (384-d),
    dual SE recalibration, head 896->512->256->6."""

    def __init__(self, num_classes=6, pretrained=True):
        super().__init__()
        vgg = timm.create_model("vgg16_bn", pretrained=pretrained, num_classes=0)
        self.stream_a = vgg
        dim_a = 512

        deit = timm.create_model("deit_small_patch16_224", pretrained=pretrained, num_classes=0)
        self.stream_b = deit
        dim_b = 384

        print(f"[*] Stream A (VGG16-BN spatial+GAP): {dim_a}-d | Stream B (DeiT-S CLS): {dim_b}-d | Combined: {dim_a+dim_b}-d")

        self.se_a = SqueezeExcitationBlock(dim_a, reduction=16)
        self.se_b = SqueezeExcitationBlock(dim_b, reduction=16)

        combined_dim = dim_a + dim_b  # 896
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(combined_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(p=0.25),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feat_map_a = self.stream_a.forward_features(x)
        feat_a = feat_map_a.mean(dim=[2, 3])
        feat_b = self.stream_b(x)
        rec_a = self.se_a(feat_a)
        rec_b = self.se_b(feat_b)
        fused = torch.cat([rec_a, rec_b], dim=1)
        return self.classifier(fused)


# ==============================================================================
# 4. LOSS, EMA & UTILITIES
# ==============================================================================
class FocalLoss(nn.Module):
    """Focal loss with per-class alpha (SINGLE weighting mechanism — no sampler)."""

    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()


class ModelEMA:
    """Deep-copy EMA — save/load the EMA weights, not the raw model."""

    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay
        for p in self.module.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_param, model_param in zip(self.module.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(model_param.data, alpha=1 - self.decay)
            # Decay BN running stats too — deepcopy'd buffers are frozen at init
            # unless decayed, so eval would otherwise use untrained statistics.
            for ema_buf, model_buf in zip(self.module.buffers(), model.buffers()):
                if ema_buf.dtype.is_floating_point:
                    ema_buf.data.mul_(self.decay).add_(model_buf.data, alpha=1 - self.decay)
                else:
                    ema_buf.data.copy_(model_buf.data)


def compute_focal_alpha(labels):
    """Inverse-frequency class weights + sadness x2.0, fear x1.2 (V6 boost)."""
    if len(labels) == 0:
        raise RuntimeError(
            "[!] Dataset is EMPTY — no images found in DATASET_DIR.\n"
            f"    Resolved DATASET_DIR = {DATASET_DIR}\n"
            "    Check that the correct Kaggle dataset is attached and the slug matches."
        )
    counts = Counter(labels)
    total = len(labels)
    missing = [CLASSES[i] for i in range(NUM_CLASSES) if counts.get(i, 0) == 0]
    if missing:
        raise RuntimeError(
            f"[!] The following classes have 0 images: {missing}\n"
            f"    DATASET_DIR = {DATASET_DIR}\n"
            "    Verify that train/valid/test subdirs contain all 6 class folders."
        )
    alpha = torch.tensor(
        [total / (NUM_CLASSES * counts[i]) for i in range(NUM_CLASSES)],
        dtype=torch.float32,
    ).to(device)
    alpha[CLASSES.index("sadness")] *= 2.0
    alpha[CLASSES.index("fear")] *= 1.2
    return alpha


# ==============================================================================
# 5. FOLD SPLITS (shared with run_all_models.py)
# ==============================================================================
print("[*] Building full dataset (train+valid+test merged) ...")
samples, labels = build_full_dataset(DATASET_DIR)
print(f"[*] Total images: {len(samples)}")

FOLD_FILE = "/kaggle/working/results/fold_id_by_path.json"
if os.path.exists(FOLD_FILE):
    with open(FOLD_FILE) as f:
        fold_id_by_path = json.load(f)
    fold_ids = [fold_id_by_path.get(s, 0) for s in samples]
    print(f"[*] Loaded fold assignment from {FOLD_FILE}")
else:
    print(f"[!] {FOLD_FILE} not found — recomputing folds (must run run_all_models.py first "
          "for identical folds).")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_ids = np.zeros(len(samples), dtype=int)
    for fold, (_, val_idx) in enumerate(skf.split(samples, labels)):
        for i in val_idx:
            fold_ids[i] = fold
    fold_ids = fold_ids.tolist()

folds = [(fold, [i for i in range(len(samples)) if fold_ids[i] != fold],
               [i for i in range(len(samples)) if fold_ids[i] == fold])
         for fold in range(N_FOLDS)]

alpha = compute_focal_alpha(labels)
print(f"[*] Focal Loss Class Weights (V6 Boost): { {CLASSES[i]: f'{alpha[i].item():.2f}' for i in range(NUM_CLASSES)} }")
loss_fn = FocalLoss(alpha=alpha, gamma=1.5)


# ==============================================================================
# 6. RESUME SUPPORT
# ==============================================================================
DONE_FILE = os.path.join(OUTPUT_DIR, "cv_done.json")
done = {}
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f:
        done = json.load(f)
    print(f"[*] Found completed folds — resuming.")


def mark_done(fold):
    done.setdefault("proposed_model", []).append(fold)
    with open(DONE_FILE, "w") as f:
        json.dump(done, f, indent=2)


# ==============================================================================
# 7. TRAINING (per fold)
# ==============================================================================
def train_stage1(model, train_idx):
    train_ds = AutismFERDataset([samples[i] for i in train_idx],
                                [labels[i] for i in train_idx], train_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              worker_init_fn=seed_worker, generator=DL_GENERATOR)

    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if "classifier" in name or "se_a" in name or "se_b" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": LEARNING_RATE * 0.1},
        {"params": head_params, "lr": LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)

    _WARMUP_EPOCHS = 10

    def _lr_lambda(epoch):
        if epoch < _WARMUP_EPOCHS:
            return float(epoch + 1) / float(_WARMUP_EPOCHS)
        progress = float(epoch - _WARMUP_EPOCHS) / float(max(1, NUM_EPOCHS - _WARMUP_EPOCHS))
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_lambda)
    ema_model = ModelEMA(model, decay=EMA_DECAY)
    scaler = GradScaler(enabled=(device.type == "cuda"))
    return train_loader, optimizer, scheduler, ema_model, scaler


@torch.no_grad()
def evaluate_tta(model, val_ds):
    """5-view TTA averaged probabilities over the validation subset.

    Batches each view over the dataset (mirrors run_all_models.evaluate_tta).
    """
    model.eval()
    tta_transforms = get_tta_transforms()
    n = len(val_ds)
    all_probs = np.zeros((n, NUM_CLASSES), dtype=np.float64)
    for t_form in tta_transforms:
        for start in range(0, n, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n)
            batch = []
            for idx in range(start, end):
                try:
                    raw = Image.open(val_ds.samples[idx]).convert("RGB")
                except Exception:
                    raw = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
                batch.append(t_form(raw))
            x = torch.stack(batch).to(device)
            with autocast(device_type=device.type,
                          dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
                logits = model(x)
            all_probs[start:end] += F.softmax(logits, dim=1).float().cpu().numpy()
    all_probs /= len(tta_transforms)
    preds = all_probs.argmax(1).tolist()
    targets = [int(t) for t in val_ds.targets]
    return preds, targets, all_probs


def train_one_epoch(model, loader, optimizer, scaler, ema_model, clip_params):
    """Run one training pass, return (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
            outputs = model(imgs)
            loss = loss_fn(outputs, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        clip_grad_norm_(clip_params, max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        ema_model.update(model)
        total_loss += loss.item() * imgs.size(0)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_model(model, val_ds):
    """Validate the EMA model on the fold's val split; return (loss, accuracy, macro-F1)."""
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    preds, targets = [], []
    loader = val_ds_transform_loader(val_ds)
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)
        val_loss += loss.item() * imgs.size(0)
        out_preds = outputs.argmax(dim=1)
        correct += (out_preds == labels).sum().item()
        total += imgs.size(0)
        preds.extend(out_preds.cpu().numpy())
        targets.extend(labels.cpu().numpy())
    f1 = f1_score(targets, preds, average="macro", zero_division=0)
    return val_loss / total, correct / total, f1


def val_ds_transform_loader(val_ds):
    loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        worker_init_fn=seed_worker, generator=DL_GENERATOR)
    return loader


def run_fold(fold, train_idx, val_idx):
    print(f"\n{'='*70}")
    print(f"  PROPOSED-MODEL | Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*70}")

    val_ds = AutismFERDataset([samples[i] for i in val_idx],
                              [labels[i] for i in val_idx], val_transform)

    # ---- Stage 1 ----
    model = CareFERModel(num_classes=NUM_CLASSES, pretrained=True).to(device)
    train_loader, optimizer, scheduler, ema_model, scaler = train_stage1(model, train_idx)

    best_val_f1 = 0.0
    patience_counter = 0
    ckpt_path = os.path.join(OUTPUT_DIR, f"proposed_model_fold{fold+1}_best.pth")
    start = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler,
                                                ema_model, clip_params=model.parameters())
        scheduler.step()
        _, val_acc, epoch_val_f1 = evaluate_model(ema_model.module, val_ds)

        elapsed = (time.time() - start) / 60
        print(f"  Epoch {epoch:3d}/{NUM_EPOCHS} | TrL {train_loss:.4f} "
              f"VaA {val_acc:.4f} F1 {epoch_val_f1:.4f} | {elapsed:.1f}min")

        if epoch_val_f1 > best_val_f1:
            best_val_f1 = epoch_val_f1
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": ema_model.module.state_dict(),
                "val_f1": best_val_f1,
            }, ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    print(f"[*] Stage 1 Complete in {(time.time()-start)/60:.1f} min | Best Val Macro F1: {best_val_f1:.4f}")

    # ---- Stage 2: frozen backbone, unfrozen head ----
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    for name, param in model.named_parameters():
        param.requires_grad = "classifier" in name or "se_a" in name or "se_b" in name
    head_params = [p for p in model.parameters() if p.requires_grad]

    optimizer_stage2 = torch.optim.AdamW(head_params, lr=1e-4, weight_decay=WEIGHT_DECAY)
    ema_model_stage2 = ModelEMA(model, decay=EMA_DECAY)
    scaler_s2 = GradScaler(enabled=(device.type == "cuda"))

    # Balanced train loader for stage 2 (single weighting — sampler only, no loss weights here).
    # Cap the weight ratio, matching run_all_models.py, to avoid over-repeating rare classes.
    train_ds_s2 = AutismFERDataset([samples[i] for i in train_idx],
                                   [labels[i] for i in train_idx], train_transform)
    counts = Counter(train_ds_s2.targets)
    inv_freq = [1.0 / counts[t] for t in train_ds_s2.targets]
    w_min = min(inv_freq)
    sample_weights = [min(w, w_min * 9.0) for w in inv_freq]
    sampler_s2 = WeightedRandomSampler(sample_weights, len(train_ds_s2), replacement=True)
    train_loader_s2 = DataLoader(train_ds_s2, batch_size=BATCH_SIZE, sampler=sampler_s2,
                                 num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                                 worker_init_fn=seed_worker, generator=DL_GENERATOR)

    best_f1_s2 = best_val_f1
    patience_s2 = 0
    start_s2 = time.time()

    for epoch in range(1, STAGE2_EPOCHS + 1):
        train_loss, _ = train_one_epoch(model, train_loader_s2, optimizer_stage2, scaler_s2,
                                        ema_model_stage2, clip_params=head_params)
        _, val_acc, epoch_val_f1 = evaluate_model(ema_model_stage2.module, val_ds)
        print(f"  Stage 2 Epoch {epoch:3d}/{STAGE2_EPOCHS} | TrL {train_loss:.4f} "
              f"VaA {val_acc:.4f} F1 {epoch_val_f1:.4f}")

        if epoch_val_f1 > best_f1_s2:
            best_f1_s2 = epoch_val_f1
            patience_s2 = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": ema_model_stage2.module.state_dict(),
                "val_f1": best_f1_s2,
            }, ckpt_path)
        else:
            patience_s2 += 1
            if patience_s2 >= PATIENCE:
                print(f"  Early stopping Stage 2 at epoch {epoch}")
                break

    print(f"[*] Stage 2 Complete in {(time.time()-start_s2)/60:.1f} min | Final Best Val Macro F1: {best_f1_s2:.4f}")

    # ---- OOF evaluation with 5-view TTA ----
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    eval_model = CareFERModel(num_classes=NUM_CLASSES, pretrained=False).to(device)
    eval_model.load_state_dict(checkpoint["model_state_dict"])
    preds, targets, probs = evaluate_tta(eval_model, val_ds)

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="macro", zero_division=0)
    prec = precision_score(targets, preds, average="macro", zero_division=0)
    rec = recall_score(targets, preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(targets, preds, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    fold_m = {"fold": fold, "accuracy": float(acc), "f1_macro": float(f1),
              "precision_macro": float(prec), "recall_macro": float(rec),
              "per_class_f1": {c: float(f) for c, f in zip(CLASSES, per_class_f1)}}
    print(f"  FOLD {fold+1} OOF (5-view TTA) — Acc: {acc:.4f} | F1: {f1:.4f}")

    # Cleanup memory to prevent Kaggle OOMs across folds
    del model, eval_model, train_loader, optimizer, scheduler, ema_model, scaler
    del train_loader_s2, optimizer_stage2, ema_model_stage2, scaler_s2
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    return fold_m, preds, targets, probs


def val_ds_transform_loader(val_ds):
    loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        worker_init_fn=seed_worker, generator=DL_GENERATOR)
    return loader


# ==============================================================================
# 8. RUN ALL FOLDS (OOF arrays + metrics persisted after every fold)
# ==============================================================================
def _load_npy(path, empty_shape):
    if os.path.exists(path):
        arr = np.load(path)
        return arr if arr.ndim > 0 else np.empty(empty_shape)
    return np.empty(empty_shape)


all_preds = _load_npy(os.path.join(OUTPUT_DIR, "oof_preds.npy"), (0,))
all_labels = _load_npy(os.path.join(OUTPUT_DIR, "oof_labels.npy"), (0,))
all_probs = _load_npy(os.path.join(OUTPUT_DIR, "oof_probs.npy"), (0, NUM_CLASSES))
fold_metrics = []
metrics_path = os.path.join(OUTPUT_DIR, "cv_metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        fold_metrics = json.load(f)

for fold, train_idx, val_idx in folds:
    if fold in done.get("proposed_model", []):
        print(f"  Fold {fold+1} already done — skipping.")
        continue
    fold_m, preds, targets, probs = run_fold(fold, train_idx, val_idx)
    fold_metrics.append(fold_m)
    all_preds = np.concatenate([all_preds, np.array(preds, dtype=int)])
    all_labels = np.concatenate([all_labels, np.array(targets, dtype=int)])
    all_probs = np.concatenate([all_probs, probs], axis=0)
    mark_done(fold)
    # persist incrementally (safe across session timeouts)
    with open(metrics_path, "w") as f:
        json.dump(fold_metrics, f, indent=2)
    np.save(os.path.join(OUTPUT_DIR, "oof_preds.npy"), all_preds)
    np.save(os.path.join(OUTPUT_DIR, "oof_labels.npy"), all_labels)
    np.save(os.path.join(OUTPUT_DIR, "oof_probs.npy"), all_probs)

# ==============================================================================
# 9. AGGREGATE + CLINICAL EVALUATION
# ==============================================================================
if all_preds.size == 0:
    raise RuntimeError("No out-of-fold predictions available — check cv_done.json consistency.")

test_preds = np.array(all_preds)
test_targets = np.array(all_labels)
test_probs = np.array(all_probs)

test_acc = accuracy_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average="macro", zero_division=0)
test_prec = precision_score(test_targets, test_preds, average="macro", zero_division=0)
test_rec = recall_score(test_targets, test_preds, average="macro", zero_division=0)

print(f"\n[+] OVERALL OUT-OF-FOLD RESULTS (5-view TTA, {len(fold_metrics)} folds):")
print(f"    Accuracy:      {test_acc:.4f}")
print(f"    Macro F1:      {test_f1:.4f}")
print(f"    Precision:     {test_prec:.4f}")
print(f"    Recall:        {test_rec:.4f}\n")
print(classification_report(test_targets, test_preds, target_names=CLASSES, digits=4))

if fold_metrics:
    print("\n  Per-fold Macro F1:", [f"{m['f1_macro']:.4f}" for m in fold_metrics])
    print(f"  Macro F1 mean +/- std: {np.mean([m['f1_macro'] for m in fold_metrics]):.4f} "
          f"+/- {np.std([m['f1_macro'] for m in fold_metrics]):.4f}")

# Distress emotions recall audit
report_dict = classification_report(test_targets, test_preds, target_names=CLASSES,
                                    output_dict=True, zero_division=0)
print("-" * 50)
print("CLINICAL SAFETY AUDIT: Distress Emotion Recall (Sensitivity)")
print("-" * 50)
for d_cls in ("anger", "fear", "sadness"):
    rec = report_dict[d_cls]["recall"]
    sup = report_dict[d_cls]["support"]
    print(f"  [{d_cls.upper():<8}] Recall: {rec*100:.1f}% (Support: {sup} images)")

# Uncertainty rejection guardrail
confidences = test_probs.max(axis=1)
rejection_rate = float(np.mean(confidences < UNCERTAINTY_THRESH)) * 100
high_conf = confidences >= UNCERTAINTY_THRESH
print(f"\n[+] CLINICAL UNCERTAINTY GUARDRAIL (Threshold = {UNCERTAINTY_THRESH*100:.0f}% Confidence):")
print(f"    High-Confidence: {high_conf.sum()} / {len(confidences)} images")
print(f"    Flagged for Caregiver Review (Low Conf): {(~high_conf).sum()} images ({rejection_rate:.1f}% Rejection Rate)")
if high_conf.sum() > 0:
    hc_acc = accuracy_score(test_targets[high_conf], test_preds[high_conf])
    hc_f1 = f1_score(test_targets[high_conf], test_preds[high_conf], average="macro", zero_division=0)
    print(f"    -> High-Confidence Subset Accuracy: {hc_acc*100:.2f}% | Macro F1: {hc_f1:.4f}")

# ==============================================================================
# 10. PUBLICATION FIGURES
# ==============================================================================
print("\n[*] Generating publication-ready figures...")
sns.set_theme(style="whitegrid", font_scale=1.1)

cm = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(8, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, cbar=True)
plt.title("Proposed-Model: Out-of-Fold Confusion Matrix (TTA K=5)", fontweight="bold", pad=15)
plt.ylabel("True Emotion Label"); plt.xlabel("Predicted Emotion Label")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "2_test_confusion_matrix.png"), dpi=300)
plt.close()

metrics_df = pd.DataFrame({
    "Class": CLASSES,
    "Precision": [report_dict[c]["precision"] for c in CLASSES],
    "Recall": [report_dict[c]["recall"] for c in CLASSES],
    "F1-Score": [report_dict[c]["f1-score"] for c in CLASSES],
}).melt(id_vars="Class", var_name="Metric", value_name="Score")
plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x="Class", y="Score", hue="Metric", palette="Set2")
plt.title("Proposed-Model: Per-Class Performance (Out-of-Fold)", fontweight="bold")
plt.ylim(0, 1.05); plt.ylabel("Score"); plt.legend(title="Metric")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "3_per_class_metrics.png"), dpi=300)
plt.close()

print(f"[*] All publication charts saved to: {OUTPUT_DIR}")

# ==============================================================================
# 11. GRAD-CAM HEATMAPS (VGG16 Stream A)
# ==============================================================================
print("\n[*] Generating Grad-CAM explainability heatmaps (VGG16 Stream A)...")


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self._activation = None
        self._fwd_handle = target_layer.register_forward_hook(self._fwd_hook)

    def _fwd_hook(self, module, inp, output):
        self._activation = output

    def generate(self, input_tensor, target_class):
        self.model.eval()
        _grad_holder = [None]
        with torch.enable_grad():
            x = input_tensor.detach().clone().requires_grad_(True)
            out = self.model(x)
            if self._activation is not None and self._activation.requires_grad:
                self._activation.retain_grad()
                _h = self._activation.register_hook(
                    lambda g: _grad_holder.__setitem__(0, g.detach()))
            else:
                return np.zeros((7, 7))
            self.model.zero_grad()
            out[0, target_class].backward()
            _h.remove()
        grads = _grad_holder[0]
        acts = self._activation.detach()
        if grads is None:
            return np.zeros((acts.shape[2], acts.shape[3]))
        weights = grads.mean(dim=[2, 3], keepdim=True)
        cam = (weights * acts).sum(dim=1).squeeze().cpu().numpy()
        cam = np.maximum(cam, 0)
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove(self):
        self._fwd_handle.remove()


# Load best fold's model for Grad-CAM (use first available checkpoint)
ckpt_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.startswith("proposed_model_fold") and f.endswith("_best.pth")])
if ckpt_files:
    eval_model = CareFERModel(num_classes=NUM_CLASSES, pretrained=False).to(device)
    eval_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, ckpt_files[0]),
                                          map_location=device)["model_state_dict"])
    eval_model.eval()

    _last_conv = None
    for _module in eval_model.stream_a.modules():
        if isinstance(_module, nn.Conv2d):
            _last_conv = _module

    if _last_conv is None:
        print("[!] No Conv2d found in stream_a — Grad-CAM skipped.")
    else:
        grad_cam = GradCAM(eval_model, _last_conv)
        _seen, _gradcam_samples = set(), []
        for path, cls in zip(samples, labels):
            if cls not in _seen:
                _seen.add(cls)
                _gradcam_samples.append((path, cls))
            if len(_seen) == NUM_CLASSES:
                break

        fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(3 * NUM_CLASSES, 6))
        fig.suptitle("Proposed-Model Grad-CAM: VGG16 Stream Discriminative Facial Regions",
                     fontsize=13, fontweight="bold")
        for col, (img_path, true_cls) in enumerate(_gradcam_samples):
            raw = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            tensor = val_transform(raw).unsqueeze(0).to(device).requires_grad_(True)
            cam = grad_cam.generate(tensor, target_class=true_cls)
            cam_up = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
                (IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
            img_np = np.array(raw) / 255.0
            heatmap = plt.cm.jet(cam_up)[:, :, :3]
            overlay = np.clip(0.5 * img_np + 0.5 * heatmap, 0, 1)
            axes[0, col].imshow(raw)
            axes[0, col].set_title(CLASSES[true_cls].capitalize(), fontsize=10, fontweight="bold")
            axes[0, col].axis("off")
            axes[1, col].imshow(overlay)
            axes[1, col].set_title("Grad-CAM", fontsize=9)
            axes[1, col].axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "4_gradcam_heatmaps.png"), dpi=300, bbox_inches="tight")
        plt.close()
        grad_cam.remove()
        print(f"[*] Grad-CAM heatmaps saved to: {OUTPUT_DIR}/4_gradcam_heatmaps.png")

print("========================================================================================")
print("  Proposed-Model Evaluation Complete! Ready for Paper Publication.")
print("========================================================================================")


[*] Auto-detected dataset at /kaggle/input/datasets/mrsohel/dataset-clean/dataset_clean
[*] Running on Device: cuda | Output Directory: /kaggle/working/results/proposed_model_proposed
[*] Dataset Directory: /kaggle/input/datasets/mrsohel/dataset-clean/dataset_clean
[*] Building full dataset (train+valid+test merged) ...
[*] Total images: 1808
[*] Loaded fold assignment from /kaggle/working/results/fold_id_by_path.json
[*] Focal Loss Class Weights (V6 Boost): {'anger': '1.80', 'fear': '5.32', 'joy': '0.36', 'natural': '1.50', 'sadness': '1.49', 'surprise': '2.41'}

  PROPOSED-MODEL | Fold 1/5


model.safetensors:   0%|          | 0.00/554M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

[*] Stream A (VGG16-BN spatial+GAP): 512-d | Stream B (DeiT-S CLS): 384-d | Combined: 896-d


RuntimeError: expected scalar type Half but found Float